# Packages and Data Instantiation

In [1]:
import pandas as pd
from pathlib import Path
from typing import Dict

def l_dir(d_p: str = "../data") -> Dict[str, pd.DataFrame]:
    """Loads all CSV files from a directory efficiently into memory.

    Args:
        d_p: Directory path containing the target files.

    Returns:
        Dictionary mapping filename stems to loaded DataFrames.
    """
    return {f.stem: pd.read_csv(f, engine="pyarrow") for f in Path(d_p).glob("*.csv")}

In [2]:
d_m = l_dir()

df_crs = d_m.get("bhp_crs")
df_evt = d_m.get("bhp_evt")
df_ff = d_m.get("bhp_ff")
df_ibs = d_m.get("bhp_ibs")
df_main = d_m.get("bhp_main")

In [3]:
class EDA:
    """Exploratory Data Analysis diagnostics for time-series feature matrices."""

    def __init__(self, m: pd.DataFrame):
        if not isinstance(m, pd.DataFrame):
            raise ValueError("The input 'm' must be a pandas DataFrame. The variable may have been overwritten or failed to load.")
        self.m = m
        self.c = m.columns

    def s_chk(self) -> pd.DataFrame:
        """Calculates column sparsity and missingness."""
        n = self.m.isnull().sum()
        p = (n / len(self.m)) * 100
        t = self.m.dtypes
        return pd.DataFrame({'n_ms': n, 'p_ms': p, 'd_ty': t}).sort_values('p_ms', ascending=False)

    def t_chk(self, d_c: str) -> Dict[str, str]:
        """Validates temporal continuity and identifies boundary limits."""
        dt = pd.to_datetime(self.m[d_c])
        return {
            'strt': str(dt.min().date()),
            'end': str(dt.max().date()),
            'n_dys': str(dt.nunique()),
            'gaps': str(len(dt) - dt.nunique())
        }

    def tgt_sts(self, t_c: str) -> pd.DataFrame:
        """Computes distribution statistics for the target variable."""
        if t_c not in self.c:
            return pd.DataFrame()
        return self.m[[t_c]].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [4]:
eda = EDA(df_main)
eda.s_chk()

,n_ms,p_ms,d_ty
keydeveventtypeid,4750,72.641077,str
headline,4750,72.641077,str
dlycaldt,0,0.000000,object
dlyret,0,0.000000,float64
at,0,0.000000,float64
lt,0,0.000000,float64
dlyvol,0,0.000000,float64
curcd,0,0.000000,str
meanest_fy1,0,0.000000,float64
meanest_fy2,0,0.000000,float64


In [5]:
eda.t_chk('dlycaldt')

{'strt': '2000-01-03', 'end': '2025-12-31', 'n_dys': '6539', 'gaps': '0'}

In [6]:
eda.tgt_sts('dlyret')

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
dlyret,6539.0,0.000714,0.023102,-0.171496,-0.061996,-0.035346,-0.011361,0.000993,0.013135,0.035248,0.05984,0.183184
